# RAG dla koncepcji miasta 15-minutowego

Ten notebook prowadzi krok po kroku przez implementację systemu RAG dla tematu:

> Miasto 15-minutowe i przestrzenna analiza dostępności usług miejskich.

Cel dydaktyczny: zobaczyć cały przepływ `PDF -> chunki -> embeddingi -> FAISS -> retrieval -> prompt -> odpowiedź -> ewaluacja`.

## 1. Instalacja zależności

Jeżeli uruchamiasz notebook w nowym środowisku, najpierw zainstaluj zależności z pliku `requirements.txt`.

```bash
pip install -r requirements.txt
```

Najważniejsze biblioteki:

- `pypdf` - odczyt tekstu z PDF,
- `sentence-transformers` - embeddingi tekstowe,
- `faiss-cpu` - lokalny vector store,
- `rouge-score` - przykładowa automatyczna ewaluacja.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.rag_system import RAGSystem

DATA_DIR = PROJECT_ROOT / "data" / "papers"
VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store" / "faiss_15_minute_city"

DATA_DIR, VECTOR_STORE_DIR

## 2. Utworzenie obiektu RAG

Parametry, które warto rozumieć:

- `chunk_size` - maksymalna długość fragmentu tekstu,
- `chunk_overlap` - ile znaków z poprzedniego chunka powtarzamy w następnym,
- `top_k` - ile najbardziej podobnych fragmentów pobieramy dla pytania,
- `embedding_model_name` - model zamieniający tekst na wektory.

In [ ]:
rag = RAGSystem(
    embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",
    chunk_size=900,
    chunk_overlap=150,
    top_k=4,
)

rag

## 3. Wczytanie dokumentów

Docelowo włóż 4-6 publikacji PDF do katalogu `data/papers/` i uruchom komórkę z `load_pdfs()`.

Na potrzeby pierwszego uruchomienia notebook ma też tryb demonstracyjny: jeśli nie ma PDF-ów, dodajemy krótkie teksty przykładowe. Dzięki temu można od razu zobaczyć cały pipeline.

In [ ]:
pdf_files = sorted(DATA_DIR.glob("*.pdf"))

if pdf_files:
    documents = rag.load_pdfs(DATA_DIR)
else:
    documents = rag.add_texts(
        [
            (
                "The 15-minute city is an urban planning concept focused on access "
                "to everyday services within a short walking or cycling distance. "
                "Typical amenities include schools, shops, health services, parks, "
                "public transport stops, and cultural facilities."
            ),
            (
                "Urban accessibility can be measured with GIS methods such as "
                "network analysis, service area analysis, travel-time isochrones, "
                "and proximity indicators. Network distance is usually more realistic "
                "than Euclidean distance because it follows the actual street network."
            ),
            (
                "Spatial equity analysis studies whether different neighbourhoods "
                "or social groups have comparable access to urban services. It can "
                "combine demographic data, amenity locations, pedestrian networks, "
                "and accessibility indicators."
            ),
        ],
        source="demo_15_minute_city.txt",
    )

len(documents), documents[:1]

## 4. Chunking

Teraz dzielimy dokumenty na mniejsze fragmenty. Każdy chunk będzie osobno embedowany i zapisany w indeksie FAISS.

W sprawozdaniu warto uzasadnić, że chunking poprawia trafność wyszukiwania, bo retriever porównuje pytanie z konkretnymi fragmentami, a nie z całymi artykułami.

In [ ]:
chunks = rag.chunk_documents()

print(f"Liczba chunków: {len(chunks)}")
print("\nPrzykładowy chunk:\n")
print(chunks[0].text[:800])

## 5. Embeddingi i FAISS

W tej komórce model `all-MiniLM-L6-v2` zamienia każdy chunk na wektor. Następnie FAISS tworzy indeks, w którym można szybko wyszukiwać najbliższe wektory.

Pierwsze uruchomienie może potrwać dłużej, bo model embeddingowy musi zostać pobrany.

In [ ]:
rag.build_vector_store()
rag.save_vector_store(VECTOR_STORE_DIR)

print(f"Zapisano vector store w: {VECTOR_STORE_DIR}")

## 6. Retrieval

Retriever pobiera najbardziej podobne fragmenty dla pytania. To jest kluczowy etap RAG: jeśli kontekst będzie nietrafiony, nawet dobry model językowy może wygenerować słabą odpowiedź.

In [ ]:
question = "How can GIS be used to measure accessibility in the 15-minute city?"
retrieved = rag.retrieve(question)

for i, item in enumerate(retrieved, start=1):
    print(f"[{i}] score={item.score:.3f}, source={item.chunk.source}, page={item.chunk.page}")
    print(item.chunk.text[:500])
    print("-" * 80)

## 7. Prompt i odpowiedź

Metoda `answer()` tworzy prompt z kontekstem. Domyślnie notebook używa prostej odpowiedzi ekstrakcyjnej, żeby działał bez klucza API.

W finalnej wersji możesz podłączyć dowolny model generatywny jako funkcję `generator(prompt) -> str`, np. OpenAI, Ollama albo lokalny pipeline Hugging Face.

In [ ]:
result = rag.answer(question)

print("PROMPT:\n")
print(result.prompt[:1500])
print("\n" + "=" * 80 + "\n")
print("ODPOWIEDŹ:\n")
print(result.answer)

## 8. Opcjonalnie: generator LLM

Poniżej znajduje się wzorzec funkcji generatora. Zostawiamy go jako przykład, bo wybór konkretnego LLM zależy od środowiska i dostępnych kluczy/API.

Najważniejsza idea: RAG nie przekazuje modelowi całej bazy dokumentów, tylko prompt z wybranymi fragmentami.

In [ ]:
# Przykład integracji z dowolnym LLM:
#
# def my_llm_generator(prompt: str) -> str:
#     response = some_llm_client.generate(prompt)
#     return response
#
# result = rag.answer(question, generator=my_llm_generator)
# print(result.answer)

## 9. Ewaluacja

Dla oceny 5.0 potrzebujemy ocenić jakość systemu. W tym projekcie pokazujemy trzy podejścia:

1. trafność pobranego kontekstu,
2. pokrycie oczekiwanych słów kluczowych w odpowiedzi,
3. opcjonalnie ROUGE-L, jeśli przygotujemy odpowiedź referencyjną.

Najważniejszą oceną jakościową dla RAG pozostaje faithfulness: czy odpowiedź rzeczywiście wynika z dostarczonego kontekstu.

In [ ]:
expected_keywords = ["GIS", "network analysis", "isochrones", "accessibility"]

context_eval = rag.evaluate_context_relevance(result.retrieved_chunks, expected_keywords)
answer_eval = rag.evaluate_keyword_coverage(result.answer, expected_keywords)

context_eval, answer_eval

## 10. Co wpisać do sprawozdania?

Po wykonaniu notebooka uzupełnij raport o:

- listę publikacji PDF,
- wybrany model embeddingowy i uzasadnienie,
- parametry chunkowania,
- informację, że FAISS przechowuje embeddingi i metadane chunków,
- przykładowe pytania i odpowiedzi,
- ocenę jakości odpowiedzi,
- wnioski: kiedy retrieval działa dobrze, a kiedy wymaga lepszych dokumentów lub parametrów.